# Out-of-sample evaluation

Thin interactive wrapper over `src/oil_models`. See `docs/` for the methodology and `reports/RESULTS.md` for the full generated results.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from oil_models import data_prep, hormuz, kilian_var as kv, scenarios, evaluate, benchmarks, regression, combine, volatility, external_data

In [2]:
full = data_prep.load_kilian_dataset()
data = full[data_prep.VAR_COLUMNS]
var_cache = {}
def var_model(w, h):
    key = w.index[-1]
    if key not in var_cache: var_cache[key] = kv.fit(w[data_prep.VAR_COLUMNS])
    return float(kv.forecast(var_cache[key], h)['ln_real_oil_price'].iloc[-1])
models = {'no-change': lambda w,h: benchmarks.no_change(w['ln_real_oil_price'], h),
          'AR(1) returns': lambda w,h: benchmarks.ar1_returns(w['ln_real_oil_price'], h),
          'VAR(24)': var_model,
          'supply-demand regression': regression.adapted_regression}
fcs = evaluate.recursive_forecasts(data, 'ln_real_oil_price', models, start='1992-01-01')
combine.equal_weight(fcs, list(models))
evaluate.scoreboard(fcs)

mspe_ratio  dm_pvalue  directional_accuracy  \
horizon model                                                                   
1       no-change                     1.0000        NaN                   NaN   
        AR(1) returns                 0.8589     0.2074                0.5833   
        VAR(24)                       1.0449     0.6643                0.5631   
        supply-demand regression      1.0115     0.4173                0.5126   
        combination                   0.8816     0.0275                0.5884   
3       no-change                     1.0000        NaN                   NaN   
        AR(1) returns                 1.0807     0.3600                0.5736   
        VAR(24)                       1.3256     0.0002                0.5254   
        supply-demand regression      1.0273     0.1920                0.5102   
        combination                   1.0127     0.6556                0.5635   
6       no-change                     1.0000        NaN                   NaN   
        AR(1) returns                 1.1102     0.0669                0.5396   
        VAR(24)                       1.4353     0.0029                0.5064   
        supply-demand regression      1.0661     0.2216                0.4399   
        combination                   1.0643     0.0903                0.5141   
12      no-change                     1.0000        NaN                   NaN   
        AR(1) returns                 1.0944     0.0900                0.5169   
        VAR(24)                       1.3777     0.0430                0.5273   
        supply-demand regression      1.1183     0.0925                0.4156   
        combination                   1.0793     0.1647                0.5351   

                                    n  
horizon model                          
1       no-change                 396  
        AR(1) returns             396  
        VAR(24)                   396  
        supply-demand regression  396  
        combination               396  
3       no-change                 394  
        AR(1) returns             394  
        VAR(24)                   394  
        supply-demand regression  394  
        combination               394  
6       no-change                 391  
        AR(1) returns             391  
        VAR(24)                   391  
        supply-demand regression  391  
        combination               391  
12      no-change                 385  
        AR(1) returns             385  
        VAR(24)                   385  
        supply-demand regression  385  
        combination               385

## GARCH(1,1)-t volatility (separate from the level forecast)

In [3]:
res = volatility.fit_garch_t(full['ln_real_oil_price'])
volatility.vol_forecast(res, 12)

1     16.627894
2     16.944714
3     17.255717
4     17.561214
5     17.861486
6     18.156794
7     18.447374
8     18.733448
9     19.015218
10    19.292874
11    19.566590
12    19.836529
Name: ann_vol_pct, dtype: float64

## External-data status (see DATA_REQUEST_PROMPT.md)

In [4]:
external_data.status()

,present,description
file,,
spot_prices_monthly.csv,False,"Monthly average Brent & WTI spot, USD/bbl, 198..."
us_inventories_monthly.csv,False,Monthly US total petroleum stocks (crude + pro...
oecd_inventories_monthly.csv,False,"Monthly OECD commercial petroleum stocks, mill..."
gasoline_monthly.csv,False,"Monthly US conventional gasoline spot, USD/bbl..."
futures_curve_monthly.csv,False,Month-end CL and BZ futures settles by months-...
world_production_monthly.csv,False,"Monthly world crude production, thousand bbl/d..."
cpi_monthly.csv,False,"US CPI (CPIAUCSL), monthly, through latest"
igrea_monthly.csv,False,"Kilian/Dallas Fed IGREA index, monthly, throug..."
